# Biohub d3ac1a 0.900: Drive-Persistent Worst12 Short-Track Ablation

This notebook uses `biohub-cell-tracking-blend-preprocessings-d3ac1a.ipynb` as the exact reference pipeline. It runs the new 8-view TTA inference once on the true worst 8 plus embryo-balanced 4 guards, stores prediction GEFFs directly on Google Drive, and compares four short-track policies without rerunning inference.

In [ ]:
from pathlib import Path
import json
import os
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
BASE_DIR = PROJECT_DIR / 'data'
REPORT_DIR = PROJECT_DIR / 'reports'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
SUPPORT_DIR = ARTIFACT_DIR / 'biohub-tracking-support-pack-50ep-v1'
TARGET_SETS_CSV = REPORT_DIR / 'targeted_validation_sets.csv'
FULL199_SCORE_CSV = REPORT_DIR / 'deepcenter_full199_local_validation_scores.csv'

# Everything important lives directly on Drive. No /content prediction cache is used.
DRIVE_WORK_DIR = REPORT_DIR / 'd3ac1a_worst12_drive_work'
DRIVE_WORK_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for required in [BASE_DIR, SUPPORT_DIR, TARGET_SETS_CSV, FULL199_SCORE_CSV]:
    if not required.exists():
        raise FileNotFoundError(required)

scores_all = pd.read_csv(FULL199_SCORE_CSV)
target_df = pd.read_csv(TARGET_SETS_CSV)
score_col = 'edge_jaccard_matched'

worst8_df = scores_all.nsmallest(8, score_col).copy()
worst8_ids = worst8_df['sample_id'].astype(str).tolist()

guard_pool_ids = set(target_df.loc[target_df['guard_high_score'].eq(1), 'sample_id'].astype(str))
guard_pool = scores_all[
    scores_all['sample_id'].astype(str).isin(guard_pool_ids) &
    ~scores_all['sample_id'].astype(str).isin(worst8_ids)
].copy()
if 'embryo_id' not in guard_pool.columns:
    guard_pool['embryo_id'] = guard_pool['sample_id'].astype(str).str.split('_').str[0]
guard4_df = pd.concat([
    guard_pool[guard_pool['embryo_id'].eq(embryo)].nlargest(2, score_col)
    for embryo in ['44b6', '6bba']
], ignore_index=True)
guard4_ids = guard4_df['sample_id'].astype(str).tolist()

assert len(worst8_ids) == 8
assert len(guard4_ids) == 4
assert not (set(worst8_ids) & set(guard4_ids))
assert guard4_df['embryo_id'].value_counts().to_dict() == {'44b6': 2, '6bba': 2}

sample_ids = sorted(worst8_ids + guard4_ids)
assert len(sample_ids) == 12
for sid in sample_ids:
    if not (BASE_DIR / 'train' / f'{sid}.zarr').exists():
        raise FileNotFoundError(BASE_DIR / 'train' / f'{sid}.zarr')
    if not (BASE_DIR / 'train' / f'{sid}.geff').exists():
        raise FileNotFoundError(BASE_DIR / 'train' / f'{sid}.geff')

print('REFERENCE NOTEBOOK: biohub-cell-tracking-blend-preprocessings-d3ac1a.ipynb (public 0.900)')
print('DRIVE_WORK_DIR:', DRIVE_WORK_DIR)
print('worst8:')
display(worst8_df[['sample_id', score_col, 'node_recall_sparse_gt', 'edge_fn']])
print('balanced guards:')
display(guard4_df[['sample_id', 'embryo_id', score_col, 'node_recall_sparse_gt', 'edge_fn']])
print('selected sample count:', len(sample_ids))
print(sample_ids)

# Point the exact d3ac1a code to Drive data/artifacts before its config cell runs.
os.environ['BIOHUB_TEST_DIR'] = str(BASE_DIR / 'train')
os.environ['BIOHUB_MODEL_ARTIFACTS'] = str(SUPPORT_DIR)
os.environ['BIOHUB_ARTIFACTS'] = str(SUPPORT_DIR)
os.environ['BIOHUB_PRIMARY_ARTIFACT_MANIFEST'] = str(SUPPORT_DIR / 'ARTIFACT_MANIFEST.json')
os.environ['BIOHUB_ALLOW_PIP_INSTALL'] = '0'
os.environ['BIOHUB_RUN_VISUAL_EDA'] = '0'
os.environ['BIOHUB_RUN_OUTPUT_DIAGNOSTICS'] = '1'
os.chdir(DRIVE_WORK_DIR)
print('cwd:', Path.cwd())


## Exact d3ac1a Winning Preset

In [ ]:
import os
BIOHUB_PRESET = "unet400_adaptive_short_track_recovery"
BIOHUB_SCORE_AXIS = "400ep graph with spatial TTA and conditional short-track recall recovery"
# Variant environment overrides. These are intentionally set before the main config cell.
os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.97"
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "0"
# Keep the high-confidence safe-division micro trim from the strongest 400ep graph setting.
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "4.66"
os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = "7.05"
os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "7.65"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"
# Adaptive recall recovery for 400ep under-count cases.
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "1"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC"] = "0.10"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN"] = "4"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB"] = "0.82"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM"] = "3.25"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC"] = "0.018"
os.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS"] = "180"
# DeepCenter is intentionally disabled in this candidate; the score-up axis is calibrated 400ep graph inference.
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "0"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "0"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = "0"
# Thresholds are retained only for audit visibility when the gate is disabled.
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.06"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD"] = "0.08"
os.environ.setdefault("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1")
print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)


## Exact d3ac1a Configuration

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_44_400ep_adaptive_short_track_recovery"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"
RUN_VISUAL_EDA = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))

# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.
USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/checkpoint_last.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "run_visual_eda": RUN_VISUAL_EDA,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

## Artifact and Dependency Setup on Drive

In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))

ensure_dependencies(ARTIFACTS)
_repo_required = [
    REPO_DIR / "scripts" / "predict_unet_transformer.py",
    REPO_DIR / WEIGHTS_RELATIVE,
]
if all(path.exists() for path in _repo_required):
    print("Reusing Drive inference repo without deleting saved predictions:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)
else:
    materialize_inference_repo(ARTIFACTS)

## Run or Resume 8-View TTA Inference

Prediction GEFFs are written directly under the Drive work directory. A complete existing 12-sample checkpoint causes inference to be skipped.

In [ ]:
# PATCH: 400ep spatial D4-style detection TTA from the strongest anchor.
_ps = REPO_DIR / "scripts" / "predict_unet_transformer.py"
_s = _ps.read_text()
_old = """        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4"""
_new = """        if cfg.det_tta:
            _nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                _nv += 1
            for _k in (1, 3):
                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))
                del imgs_rot, det_rot
                _nv += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            _nv += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            _nv += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / _nv"""
if _old in _s:
    _ps.write_text(_s.replace(_old, _new))
    print("TTA patch applied (400ep spatial D4-style)")
elif "for _k in (1, 3):" in _s and "_nv += 1" in _s:
    print("TTA 8-view patch already present in Drive repo")
else:
    raise RuntimeError("Could not apply or verify the required d3ac1a 8-view TTA patch")

def list_test_stems() -> list[str]:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory does not exist: {TEST_DIR}")
    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No test .zarr files found in {TEST_DIR}")
    return stems


test_stems = list(sample_ids)
print(f'Selected {len(test_stems)} validation videos')
print(test_stems)

splits_path = REPO_DIR / 'kaggle_test_splits_50ep.json'
splits_path.parent.mkdir(parents=True, exist_ok=True)
splits_path.write_text(json.dumps([{'split': 0, 'train': [], 'test': test_stems}], indent=2))

predict_cmd = [
    sys.executable,
    'scripts/predict_unet_transformer.py',
    '--data-dir', str(TEST_DIR),
    '--splits', str(splits_path.name),
    '--split', '0',
    '--weights', WEIGHTS_RELATIVE,
    '--unet-batch-size', str(UNET_BATCH_SIZE),
    '--det-threshold', str(DET_THRESHOLD),
    '--ilp-edge-weight', str(ILP_EDGE_WEIGHT),
    '--ilp-appearance-weight', str(ILP_APPEARANCE_WEIGHT),
    '--ilp-disappearance-weight', str(ILP_DISAPPEARANCE_WEIGHT),
    '--ilp-division-weight', str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    predict_cmd.append('--use-ilp')
if SLICE:
    predict_cmd.extend(['--slice', SLICE])

pred_dir = REPO_DIR / 'predictions' / 'unknown' / METHOD / 'split_0'
pred_dir.mkdir(parents=True, exist_ok=True)
required_geff_parts = [
    'nodes/ids', 'nodes/props/t/values', 'nodes/props/z/values',
    'nodes/props/y/values', 'nodes/props/x/values', 'edges/ids',
]

def geff_complete(path: Path) -> bool:
    return path.exists() and all((path / rel).exists() for rel in required_geff_parts)

complete_before = [sid for sid in test_stems if geff_complete(pred_dir / f'{sid}.geff')]
print('Drive prediction checkpoint:', pred_dir.resolve())
print(f'complete predictions before run: {len(complete_before)}/{len(test_stems)}')

if len(complete_before) == len(test_stems):
    print('All selected d3ac1a predictions already exist on Drive; inference skipped.')
    predict_seconds = 0.0
else:
    start_time = time.time()
    print(' '.join(predict_cmd))
    subprocess.run(predict_cmd, cwd=REPO_DIR, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)
    predict_seconds = time.time() - start_time
    print(f'Prediction completed in {predict_seconds / 60:.2f} minutes')

complete_after = [sid for sid in test_stems if geff_complete(pred_dir / f'{sid}.geff')]
missing_after = sorted(set(test_stems) - set(complete_after))
print(f'complete Drive predictions after run: {len(complete_after)}/{len(test_stems)}')
print('missing/incomplete:', missing_after)
assert not missing_after, 'Some d3ac1a prediction GEFFs are missing or incomplete.'

checkpoint_manifest = REPORT_DIR / 'd3ac1a_worst8_guard4_prediction_checkpoint_manifest.csv'
pd.DataFrame({
    'sample_id': test_stems,
    'geff_path': [str(pred_dir / f'{sid}.geff') for sid in test_stems],
    'complete': [geff_complete(pred_dir / f'{sid}.geff') for sid in test_stems],
}).to_csv(checkpoint_manifest, index=False)
print('saved checkpoint manifest:', checkpoint_manifest)


## Prepare Four Short-Track Variants

In [ ]:
from pathlib import Path
import pandas as pd
import time
from tqdm.auto import tqdm

CONFIG_CODE = 'from __future__ import annotations\n\nimport csv\nimport importlib.util\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport zipfile\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOMPETITION = "biohub-cell-tracking-during-development"\nCOMP_DIR_CANDIDATES = [\n    Path(f"/kaggle/input/competitions/{COMPETITION}"),\n    Path(f"/kaggle/input/{COMPETITION}"),\n]\nCOMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])\n_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()\nTEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"\n\nWORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")\nREPO_DIR = WORKING_DIR / "tracking_repo"\nSUBMISSION_PATH = WORKING_DIR / "submission.csv"\nRUN_STATS_PATH = WORKING_DIR / "run_stats.csv"\n\nMETHOD = "unet_transformer"\nWEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"\nEXPERIMENT_TAG = "selected_44_400ep_adaptive_short_track_recovery"\nTARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")\nPRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(\n    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",\n    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",\n))\nALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"\n\nDET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))\nUNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))\nUSE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"\nILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))\nILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))\nILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))\nILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))\n\n# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.\nSLICE = os.environ.get("BIOHUB_SLICE", "").strip()\n\n# If dependencies are not already installed and no offline wheels are attached,\n# this controls whether the notebook attempts PyPI installation.\nALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"\nRUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"\nRUN_VISUAL_EDA = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "1") != "0"\n\n# Output-level graph post-processing.\nOUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))\nOUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"\nOUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"\nOUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"\nOUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"\nOUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"\nMOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))\nMOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))\nMOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))\nMOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))\nMOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))\n\nOUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"\nDIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))\nDIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))\nDIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"\nOUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"\nGAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))\nGAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))\nGAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"\nGAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))\nGAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))\nGAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))\nGAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"\nGAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))\nGAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))\nGAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))\n\nOUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"\nOUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))\nOUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"\nADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"\nSHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))\nSHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))\nSHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))\nSHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))\nSHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))\nSHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))\n\nOUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"\nOUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))\nOUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))\n\nOUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"\nGAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))\nGAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))\nGAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))\nGAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))\nGAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"\nGAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))\n\nOUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"\nSAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))\nSAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))\nSAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))\nSAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))\nSAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))\n\n# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.\nUSE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"\nREQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"\nDEEPCENTER_MANIFEST_DEFAULT = os.environ.get(\n    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",\n    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",\n)\nDEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(\n    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",\n    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt",\n)\nDEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/checkpoint_last.pt")\nDEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"\nDEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"\nDEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))\nDEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))\nDEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))\nDEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))\nDEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))\n\nCONFIG_DISPLAY = {\n    "experiment_tag": EXPERIMENT_TAG,\n    "method": METHOD,\n    "weights": WEIGHTS_RELATIVE,\n    "target_artifact_slug": TARGET_ARTIFACT_SLUG,\n    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),\n    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,\n    "det_threshold": DET_THRESHOLD,\n    "unet_batch_size": UNET_BATCH_SIZE,\n    "use_ilp": USE_ILP,\n    "ilp_edge_weight": ILP_EDGE_WEIGHT,\n    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,\n    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,\n    "ilp_division_weight": ILP_DIVISION_WEIGHT,\n    "slice": SLICE,\n    "allow_pip_install": ALLOW_PIP_INSTALL,\n    "run_visual_eda": RUN_VISUAL_EDA,\n    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,\n    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,\n    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,\n    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,\n    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,\n    "output_motion_relink": OUTPUT_MOTION_RELINK,\n    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,\n    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,\n    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,\n    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,\n    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,\n    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,\n    "div_parent_max_um": DIV_PARENT_MAX_UM,\n    "div_sister_max_um": DIV_SISTER_MAX_UM,\n    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,\n    "output_gap_close": OUTPUT_GAP_CLOSE,\n    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,\n    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),\n    "gap_close_um": GAP_CLOSE_UM,\n    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,\n    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,\n    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,\n    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,\n    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,\n    "gap_refine_win_z": GAP_REFINE_WIN_Z,\n    "gap_refine_win_yx": GAP_REFINE_WIN_YX,\n    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,\n    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,\n    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,\n    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,\n    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,\n    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,\n    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,\n    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,\n    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,\n    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,\n    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,\n    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,\n    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,\n    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,\n    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,\n    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,\n    "gap2_max_step_um": GAP2_MAX_STEP_UM,\n    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,\n    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,\n    "gap2_require_context": GAP2_REQUIRE_CONTEXT,\n    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,\n    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,\n    "safe_div_max_um": SAFE_DIV_MAX_UM,\n    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,\n    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,\n    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,\n    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,\n    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,\n    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,\n    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,\n    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,\n    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,\n    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,\n}\n\nprint("Biohub learned UNet + node-transformer + ILP submission")\nprint("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())\nprint("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())\nprint(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))'
BUILD_SUBMISSION_CODE = 'import tracksdata as td\nimport numpy as np\nimport blosc2\nfrom scipy.optimize import linear_sum_assignment\n\nSUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\nCSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]\nVOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)\n\n\ndef graph_from_geff(path: Path):\n    graph = td.graph.IndexedRXGraph.from_geff(path)\n    return graph[0] if isinstance(graph, tuple) else graph\n\n\ndef edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:\n    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]\n    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]\n    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:\n    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]\n    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]\n    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef node_point(node: dict[str, object]) -> tuple[float, float, float]:\n    return (float(node["z"]), float(node["y"]), float(node["x"]))\n\n\ndef edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:\n    prob = edge.get("edge_prob")\n    prob_value = float(prob) if prob is not None else 0.0\n    return prob_value, -float(edge["distance_um"])\n\n\ndef _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:\n    return max(nodes_by_id) + 1 if nodes_by_id else 1\n\n\n\ndef read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:\n    if t in frame_cache:\n        return frame_cache[t]\n    zarr_path = TEST_DIR / f"{dataset}.zarr"\n    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())\n    shape = tuple(int(v) for v in meta["shape"])\n    dtype = np.dtype(meta["data_type"])\n    frame_shape = shape[1:]\n    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"\n    try:\n        raw = chunk_path.read_bytes()\n        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)\n        if arr.size == int(np.prod(frame_shape)):\n            frame = arr.reshape(frame_shape).copy()\n            frame_cache[t] = frame\n            return frame\n    except Exception:\n        pass\n    import zarr\n    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])\n    frame_cache[t] = frame\n    return frame\n\n\ndef refine_synthetic_midpoint(\n    dataset: str | None,\n    t: int,\n    midpoint: tuple[float, float, float],\n    frame_cache: dict[int, np.ndarray],\n    stats: dict[str, int],\n) -> tuple[float, float, float]:\n    if not GAP_REFINE_SYNTHETIC or dataset is None:\n        return midpoint\n    try:\n        frame = read_test_frame(dataset, t, frame_cache)\n        z, y, x = [int(round(v)) for v in midpoint]\n        z0 = max(0, z - GAP_REFINE_WIN_Z)\n        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)\n        y0 = max(0, y - GAP_REFINE_WIN_YX)\n        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)\n        x0 = max(0, x - GAP_REFINE_WIN_YX)\n        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)\n        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)\n        if patch.size == 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        baseline = float(np.percentile(patch, 20.0))\n        weights = np.maximum(patch - baseline, 0.0)\n        total = float(weights.sum())\n        if total <= 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]\n        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]\n        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]\n        refined = (\n            float((weights * zz).sum() / total),\n            float((weights * yy).sum() / total),\n            float((weights * xx).sum() / total),\n        )\n        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:\n            stats["gap_refine_rejected_shift"] += 1\n            return midpoint\n        stats["gap_refined_synthetic"] += 1\n        return refined\n    except Exception:\n        stats["gap_refine_failed"] += 1\n        return midpoint\n\n\n\ndef _dc_pool_frame_xy(volume: np.ndarray, factor: int) -> np.ndarray:\n    if factor <= 1:\n        return volume.astype(np.float32, copy=False)\n    z, y, x = volume.shape\n    y2 = (y // factor) * factor\n    x2 = (x // factor) * factor\n    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)\n    return cropped.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))\n\n\ndef _dc_normalize_dynamic_range(volume: np.ndarray, cfg: object) -> np.ndarray:\n    vol = np.asarray(volume, dtype=np.float32)\n    lo = float(np.percentile(vol, float(getattr(cfg, "norm_lo_pct", 50.0))))\n    hi = float(np.percentile(vol, float(getattr(cfg, "norm_hi_pct", 99.5))))\n    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:\n        return np.zeros_like(vol, dtype=np.float32)\n    ratio = (vol - lo) / (hi - lo)\n    return np.clip(\n        ratio,\n        float(getattr(cfg, "norm_clip_lo", -0.5)),\n        float(getattr(cfg, "norm_clip_hi", 6.0)),\n    ).astype(np.float32)\n\n\ndef _dc_manifest_weight_paths(manifest_path: Path) -> list[Path]:\n    if not manifest_path.exists():\n        return []\n    try:\n        manifest = json.loads(manifest_path.read_text())\n    except Exception as exc:\n        print("Could not read DeepCenter manifest:", manifest_path, type(exc).__name__, exc)\n        return []\n    root = manifest_path.parent\n    sections: list[dict[str, object]] = []\n    for section in [\n        manifest.get("model", {}),\n        manifest.get("models", {}).get("full_frame_center", {}) if isinstance(manifest.get("models", {}), dict) else {},\n        manifest.get("full_frame_center", {}),\n    ]:\n        if isinstance(section, dict):\n            sections.append(section)\n    candidates: list[Path] = []\n    for section in sections:\n        for key in ("weight_path", "path"):\n            rel = section.get(key)\n            if isinstance(rel, str) and rel:\n                candidates.append(root / rel)\n        for key in ("last_checkpoint", "best_checkpoint"):\n            item = section.get(key)\n            if isinstance(item, dict):\n                rel = item.get("path")\n                if isinstance(rel, str) and rel:\n                    candidates.append(root / rel)\n    for name in ("checkpoint_last.pt", "best.pt", "last.pt"):\n        candidates.append(root / "weights" / "full_frame_center" / name)\n        candidates.append(root / name)\n    candidates.append(root / DEEPCENTER_RELATIVE)\n    return candidates\n\n\ndef _dc_checkpoint_candidates() -> list[Path]:\n    candidates: list[Path] = []\n    explicit = os.environ.get("BIOHUB_DEEPCENTER_CHECKPOINT", DEEPCENTER_CHECKPOINT_DEFAULT).strip()\n    if explicit:\n        candidates.append(Path(explicit))\n    manifest_explicit = os.environ.get("BIOHUB_DEEPCENTER_MANIFEST", DEEPCENTER_MANIFEST_DEFAULT).strip()\n    if manifest_explicit:\n        candidates.extend(_dc_manifest_weight_paths(Path(manifest_explicit)))\n\n    input_root = Path("/kaggle/input")\n    preferred_dirs = [\n        Path("/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1"),\n        Path("/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1"),\n    ]\n    for directory in preferred_dirs:\n        candidates.extend(_dc_manifest_weight_paths(directory / "ARTIFACT_MANIFEST.json"))\n        for name in ("checkpoint_last.pt", "best.pt", "last.pt"):\n            candidates.append(directory / "weights" / "full_frame_center" / name)\n            candidates.append(directory / name)\n    if input_root.exists():\n        for name in ("checkpoint_last.pt", "best.pt", "last.pt"):\n            candidates.extend(sorted(input_root.glob(f"**/full_frame_center/**/{name}")))\n\n    seen: set[Path] = set()\n    out: list[Path] = []\n    for path in candidates:\n        path = path.expanduser()\n        try:\n            key = path.resolve() if path.exists() else path\n        except Exception:\n            key = path\n        if key in seen:\n            continue\n        seen.add(key)\n        out.append(path)\n    return out\n\n\ntry:\n    import torch\nexcept Exception as _dc_torch_error:\n    torch = None\n\n\nif torch is not None:\n    class _DCConvBlock3d(torch.nn.Module):\n        def __init__(self, in_channels: int, out_channels: int) -> None:\n            super().__init__()\n            groups = min(8, out_channels)\n            self.block = torch.nn.Sequential(\n                torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),\n                torch.nn.GroupNorm(groups, out_channels),\n                torch.nn.SiLU(inplace=True),\n                torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),\n                torch.nn.GroupNorm(groups, out_channels),\n                torch.nn.SiLU(inplace=True),\n            )\n\n        def forward(self, x):\n            return self.block(x)\n\n\n    class _DCDeepCenterUNet3D(torch.nn.Module):\n        def __init__(self, in_channels: int = 1, base_channels: int = 24) -> None:\n            super().__init__()\n            c = int(base_channels)\n            self.enc1 = _DCConvBlock3d(in_channels, c)\n            self.down1 = torch.nn.MaxPool3d(2, 2)\n            self.enc2 = _DCConvBlock3d(c, c * 2)\n            self.down2 = torch.nn.MaxPool3d(2, 2)\n            self.enc3 = _DCConvBlock3d(c * 2, c * 4)\n            self.down3 = torch.nn.MaxPool3d(2, 2)\n            self.bottleneck = _DCConvBlock3d(c * 4, c * 8)\n            self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)\n            self.dec3 = _DCConvBlock3d(c * 8, c * 4)\n            self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)\n            self.dec2 = _DCConvBlock3d(c * 4, c * 2)\n            self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)\n            self.dec1 = _DCConvBlock3d(c * 2, c)\n            self.head = torch.nn.Conv3d(c, 1, 1)\n\n        def forward(self, x):\n            e1 = self.enc1(x)\n            e2 = self.enc2(self.down1(e1))\n            e3 = self.enc3(self.down2(e2))\n            b = self.bottleneck(self.down3(e3))\n            d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))\n            d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))\n            d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))\n            return self.head(d1)\nelse:\n    _DCConvBlock3d = None\n    _DCDeepCenterUNet3D = None\n\ndef load_deepcenter_veto_detector() -> dict[str, object] | None:\n    if not USE_DEEPCENTER_VETO:\n        print("DeepCenter add-only repair gate disabled by configuration.")\n        return None\n    if torch is None:\n        if REQUIRE_DEEPCENTER_VETO:\n            raise ImportError("torch is required for DeepCenter add-only repair gate")\n        print("DeepCenter add-only repair gate skipped because torch is unavailable.")\n        return None\n    from types import SimpleNamespace\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    load_errors: list[str] = []\n    for checkpoint_path in _dc_checkpoint_candidates():\n        if not checkpoint_path.exists():\n            continue\n        try:\n            print("Trying DeepCenter add-only gate checkpoint:", checkpoint_path)\n            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n            if not isinstance(checkpoint, dict) or "model_state" not in checkpoint:\n                raise ValueError("checkpoint has no model_state")\n            cfg = SimpleNamespace(**checkpoint.get("config", {}))\n            model = _DCDeepCenterUNet3D(base_channels=int(getattr(cfg, "base_channels", 24)))\n            model.load_state_dict(checkpoint["model_state"])\n            model.to(device)\n            model.eval()\n            print("Loaded DeepCenter add-only gate checkpoint:", checkpoint_path)\n            print("DeepCenter checkpoint epoch:", checkpoint.get("epoch"), "best_score:", checkpoint.get("best_score"))\n            return {\n                "model": model,\n                "cfg": cfg,\n                "device": device,\n                "path": checkpoint_path,\n                "torch": torch,\n            }\n        except Exception as exc:\n            load_errors.append(f"{checkpoint_path}: {type(exc).__name__}: {exc}")\n            print("Skipping incompatible DeepCenter checkpoint:", checkpoint_path, "|", type(exc).__name__, exc)\n    message = "No usable DeepCenter checkpoint found for add-only repair gate."\n    if REQUIRE_DEEPCENTER_VETO:\n        checked = "\\n".join(str(p) for p in _dc_checkpoint_candidates()[:80])\n        errors = "\\n".join(load_errors[-20:])\n        raise FileNotFoundError(message + "\\nChecked:\\n" + checked + ("\\nLoad errors:\\n" + errors if errors else ""))\n    print(message)\n    return None\n\n\ndef _dc_cache_trim(cache: dict[tuple[str, int], np.ndarray]) -> None:\n    limit = max(1, int(DEEPCENTER_SCORE_CACHE_MAX_FRAMES))\n    while len(cache) > limit:\n        cache.pop(next(iter(cache)))\n\n\ndef deepcenter_heatmap_for_frame(\n    dataset: str,\n    t: int,\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> np.ndarray | None:\n    if detector_bundle is None:\n        return None\n    key = (dataset, int(t))\n    cached = heatmap_cache.get(key)\n    if cached is not None:\n        return cached\n    model = detector_bundle["model"]\n    cfg = detector_bundle["cfg"]\n    device = detector_bundle["device"]\n    torch_mod = detector_bundle["torch"]\n    pool_factor = int(getattr(cfg, "pool_factor", 4))\n    volume = read_test_frame(dataset, int(t), frame_cache)\n    pooled = _dc_pool_frame_xy(volume, pool_factor)\n    image = _dc_normalize_dynamic_range(pooled, cfg)\n    with torch_mod.no_grad():\n        tensor = torch_mod.from_numpy(image[None, None, ...]).to(device=device, dtype=torch_mod.float32)\n        heatmap = torch_mod.sigmoid(model(tensor))[0, 0].detach().cpu().numpy().astype(np.float32, copy=False)\n    heatmap_cache[key] = heatmap\n    _dc_cache_trim(heatmap_cache)\n    return heatmap\n\n\ndef deepcenter_score_point(\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> float | None:\n    if not USE_DEEPCENTER_VETO or detector_bundle is None or dataset is None:\n        return None\n    heatmap = deepcenter_heatmap_for_frame(dataset, int(t), detector_bundle, frame_cache, heatmap_cache)\n    if heatmap is None or heatmap.size == 0:\n        return None\n    cfg = detector_bundle["cfg"]\n    pool_factor = int(getattr(cfg, "pool_factor", 4))\n    z = int(round(float(point[0])))\n    y = int(round(float(point[1]) / max(pool_factor, 1)))\n    x = int(round(float(point[2]) / max(pool_factor, 1)))\n    z0, z1 = max(0, z - DEEPCENTER_SCORE_WIN_Z), min(heatmap.shape[0], z + DEEPCENTER_SCORE_WIN_Z + 1)\n    y0, y1 = max(0, y - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[1], y + DEEPCENTER_SCORE_WIN_YX + 1)\n    x0, x1 = max(0, x - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[2], x + DEEPCENTER_SCORE_WIN_YX + 1)\n    patch = heatmap[z0:z1, y0:y1, x0:x1]\n    if patch.size == 0:\n        return None\n    score = float(np.max(patch))\n    return score if np.isfinite(score) else None\n\n\ndef deepcenter_accept_repair_point(\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n    stats: dict[str, int],\n    prefix: str,\n    threshold: float,\n) -> bool:\n    if not USE_DEEPCENTER_VETO:\n        return True\n    if detector_bundle is None or dataset is None:\n        stats[f"deepcenter_{prefix}_missing"] += 1\n        return True\n    stats[f"deepcenter_{prefix}_checked"] += 1\n    score = deepcenter_score_point(dataset, int(t), point, detector_bundle, frame_cache, heatmap_cache)\n    if score is None:\n        stats[f"deepcenter_{prefix}_missing"] += 1\n        return True\n    if score < float(threshold):\n        stats[f"deepcenter_{prefix}_rejected"] += 1\n        return False\n    stats[f"deepcenter_{prefix}_accepted"] += 1\n    return True\n\ndef _position_um(node: dict[str, object]) -> np.ndarray:\n    return np.array(\n        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],\n        dtype=np.float64,\n    )\n\n\ndef motion_relink_edges(\n    nodes_by_id: dict[int, dict[str, object]],\n    stats: dict[str, int],\n    learned_edge_probs: dict[tuple[int, int], float] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_MOTION_RELINK or not nodes_by_id:\n        return []\n\n    learned_edge_probs = learned_edge_probs or {}\n\n    def learned_prob(source_id: int, target_id: int) -> float:\n        value = learned_edge_probs.get((source_id, target_id), 0.0)\n        try:\n            value = float(value)\n        except (TypeError, ValueError):\n            return 0.0\n        if not np.isfinite(value):\n            return 0.0\n        if value < 0.0 or value > 1.0:\n            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n        return float(np.clip(value, 0.0, 1.0))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n    for ids in ids_by_t.values():\n        ids.sort()\n\n    frame_sizes = [len(ids) for ids in ids_by_t.values()]\n    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:\n        stats["motion_relink_skipped_large_frame"] = 1\n        return []\n\n    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}\n    predecessor_position_um: dict[int, np.ndarray] = {}\n    selected_edges: list[dict[str, object]] = []\n\n    def assign_pass(\n        source_ids: list[int],\n        target_ids: list[int],\n        gate_um: float,\n    ) -> list[tuple[int, int, float, float, float]]:\n        if not source_ids or not target_ids:\n            return []\n        big = gate_um * 1000.0 + 1.0\n        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)\n        raw_dist = np.full_like(cost, np.inf)\n        motion_dist = np.full_like(cost, np.inf)\n        prob_matrix = np.zeros_like(cost)\n        for i, source_id in enumerate(source_ids):\n            source_pos = position_um[source_id]\n            prev_pos = predecessor_position_um.get(source_id)\n            if prev_pos is None:\n                predicted = source_pos\n            else:\n                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\n            for j, target_id in enumerate(target_ids):\n                target_pos = position_um[target_id]\n                raw = float(np.linalg.norm(target_pos - source_pos))\n                if raw > gate_um:\n                    continue\n                motion = float(np.linalg.norm(target_pos - predicted))\n                prob = learned_prob(source_id, target_id)\n                raw_dist[i, j] = raw\n                motion_dist[i, j] = motion\n                prob_matrix[i, j] = prob\n                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob\n        row_ind, col_ind = linear_sum_assignment(cost)\n        matches: list[tuple[int, int, float, float, float]] = []\n        for r, c in zip(row_ind, col_ind):\n            if cost[r, c] >= big:\n                continue\n            matches.append((\n                source_ids[int(r)],\n                target_ids[int(c)],\n                float(raw_dist[r, c]),\n                float(motion_dist[r, c]),\n                float(prob_matrix[r, c]),\n            ))\n        return matches\n\n    times = sorted(ids_by_t)\n    for t in times:\n        source_ids = ids_by_t.get(t, [])\n        target_ids = ids_by_t.get(t + 1, [])\n        if not source_ids or not target_ids:\n            continue\n        unmatched_sources = set(source_ids)\n        unmatched_targets = set(target_ids)\n        frame_matches: list[tuple[int, int, float, float, str, float]] = []\n        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):\n            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]\n            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]\n            matches = assign_pass(pass_sources, pass_targets, gate_um)\n            for source_id, target_id, raw, motion, prob in matches:\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\n                    continue\n                unmatched_sources.remove(source_id)\n                unmatched_targets.remove(target_id)\n                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))\n                if pass_name == "tight":\n                    stats["motion_relink_tight_edges"] += 1\n                else:\n                    stats["motion_relink_relaxed_edges"] += 1\n        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:\n            selected_edges.append({\n                "source_id": source_id,\n                "target_id": target_id,\n                "edge_prob": prob,\n                "distance_um": raw,\n                "motion_distance_um": motion,\n                "motion_relinked": 1,\n                "motion_pass": pass_name,\n            })\n            predecessor_position_um[target_id] = position_um[source_id]\n        stats["motion_relink_frames"] += 1\n\n    stats["motion_relink_edges"] = len(selected_edges)\n    return selected_edges\n\ndef close_single_frame_gaps(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    incident = outgoing | incoming\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    isolated_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incident:\n            isolated_by_t.setdefault(t, []).append(node_id)\n\n    max_synthetic = min(\n        GAP_CLOSE_MAX_ADDED_ABS,\n        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,\n    )\n    next_id = _next_node_id(nodes_by_id)\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n    used_starts: set[int] = set()\n    used_isolated: set[int] = set()\n    synthetic_added = 0\n    new_edges: list[dict[str, object]] = []\n\n    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)\n    stats["gap_close_effective_max_gap"] = effective_gap_max\n    for gap in range(1, effective_gap_max + 1):\n        for t, end_ids in sorted(ends_by_t.items()):\n            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]\n            if not end_ids or not start_ids:\n                continue\n\n            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]\n            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]\n            threshold_um = GAP_CLOSE_UM * (gap + 1)\n            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)\n            for i, ep in enumerate(end_points):\n                for j, sp in enumerate(start_points):\n                    d[i, j] = point_distance_um(ep, sp)\n            stats["gap_candidates"] += int((d <= threshold_um).sum())\n            if not np.isfinite(d).any():\n                continue\n\n            big = threshold_um * 1000.0 + 1.0\n            cost = np.where(d <= threshold_um, d, big)\n            row_ind, col_ind = linear_sum_assignment(cost)\n            for r, c in zip(row_ind, col_ind):\n                if d[r, c] > threshold_um:\n                    continue\n                source_id = end_ids[int(r)]\n                target_id = start_ids[int(c)]\n                if source_id in outgoing or target_id in used_starts:\n                    continue\n\n                source = nodes_by_id[source_id]\n                target = nodes_by_id[target_id]\n                mid_t = int(source["t"]) + gap\n                mid_point = (\n                    (float(source["z"]) + float(target["z"])) / 2.0,\n                    (float(source["y"]) + float(target["y"])) / 2.0,\n                    (float(source["x"]) + float(target["x"])) / 2.0,\n                )\n\n                middle_id: int | None = None\n                middle_reused = False\n                if GAP_CLOSE_REUSE_EXISTING:\n                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]\n                    if candidates:\n                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]\n                        best_idx = int(np.argmin(distances))\n                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:\n                            middle_id = candidates[best_idx]\n                            middle_reused = True\n\n                if middle_id is None:\n                    if synthetic_added >= max_synthetic:\n                        stats["gap_skipped_node_cap"] += 1\n                        continue\n                    middle_id = next_id\n                    next_id += 1\n                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)\n                    nodes_by_id[middle_id] = {\n                        "node_id": middle_id,\n                        "t": mid_t,\n                        "z": refined_point[0],\n                        "y": refined_point[1],\n                        "x": refined_point[2],\n                        "gap_synthetic": 1,\n                    }\n                    synthetic_added += 1\n                    stats["gap_inserted_synthetic"] += 1\n\n                middle = nodes_by_id[middle_id]\n                if DEEPCENTER_GAP_VETO and not deepcenter_accept_repair_point(\n                    dataset,\n                    mid_t,\n                    node_point(middle),\n                    deepcenter_bundle,\n                    frame_cache,\n                    deepcenter_cache,\n                    stats,\n                    "gap",\n                    DEEPCENTER_GAP_THRESHOLD,\n                ):\n                    if int(middle.get("gap_synthetic", 0)) == 1:\n                        nodes_by_id.pop(middle_id, None)\n                        synthetic_added = max(0, synthetic_added - 1)\n                        stats["gap_inserted_synthetic"] = max(0, stats["gap_inserted_synthetic"] - 1)\n                    continue\n                if middle_reused:\n                    used_isolated.add(middle_id)\n                    stats["gap_reused_existing"] += 1\n\n                e1 = {\n                    "source_id": source_id,\n                    "target_id": middle_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(source, middle),\n                    "gap_closed": 1,\n                }\n                e2 = {\n                    "source_id": middle_id,\n                    "target_id": target_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(middle, target),\n                    "gap_closed": 1,\n                }\n                new_edges.extend([e1, e2])\n                outgoing.add(source_id)\n                incoming.add(middle_id)\n                outgoing.add(middle_id)\n                incoming.add(target_id)\n                used_starts.add(target_id)\n                stats["gap_pairs_selected"] += 1\n                stats["gap_added_edges"] += 2\n\n    if new_edges:\n        edges = [*edges, *new_edges]\n    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]\n    return nodes_by_id, edges\n\n\ndef _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_source: dict[int, list[int]] = {}\n    for edge in edges:\n        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))\n    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}\n\n\ndef _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_target: dict[int, list[int]] = {}\n    for edge in edges:\n        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))\n    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}\n\n\ndef recover_strict_gap2(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    predecessor = _single_predecessor_map(edges)\n    successor = _single_successor_map(edges)\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n\n    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))\n    proposals: list[tuple[float, int, int, int, float]] = []\n\n    def pos_um(node_id: int) -> np.ndarray:\n        node = nodes_by_id[node_id]\n        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)\n\n    for t, end_ids in sorted(ends_by_t.items()):\n        start_ids = starts_by_t.get(t + 3, [])\n        if not end_ids or not start_ids:\n            continue\n        for end_id in end_ids:\n            end_pos = pos_um(end_id)\n            for start_id in start_ids:\n                start_pos = pos_um(start_id)\n                dist = float(np.linalg.norm(start_pos - end_pos))\n                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:\n                    continue\n                step = (start_pos - end_pos) / 3.0\n                context_penalty = 0.0\n                if GAP2_REQUIRE_CONTEXT:\n                    ok_context = False\n                    prev_id = predecessor.get(end_id)\n                    if prev_id is not None:\n                        prev_step = end_pos - pos_um(prev_id)\n                        prev_norm = float(np.linalg.norm(prev_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if prev_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    next_id = successor.get(start_id)\n                    if next_id is not None:\n                        next_step = pos_um(next_id) - start_pos\n                        next_norm = float(np.linalg.norm(next_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if next_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    if not ok_context:\n                        continue\n                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))\n\n    proposals.sort(key=lambda item: item[0])\n    stats["gap2_candidates"] = len(proposals)\n    if not proposals:\n        return nodes_by_id, edges\n\n    selected: list[tuple[float, int, int, int, float]] = []\n    used_ends: set[int] = set()\n    used_starts: set[int] = set()\n    per_frame_count: dict[int, int] = {}\n    for proposal in proposals:\n        if len(selected) >= cap:\n            stats["gap2_skipped_cap"] += 1\n            break\n        _, end_id, start_id, t, _ = proposal\n        if end_id in used_ends or start_id in used_starts:\n            continue\n        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))\n        if per_frame_count.get(t, 0) >= frame_cap:\n            continue\n        selected.append(proposal)\n        used_ends.add(end_id)\n        used_starts.add(start_id)\n        per_frame_count[t] = per_frame_count.get(t, 0) + 1\n\n    if not selected:\n        return nodes_by_id, edges\n\n    next_node_id = _next_node_id(nodes_by_id)\n    frame_cache: dict[int, np.ndarray] = {}\n    new_edges: list[dict[str, object]] = []\n    for _, end_id, start_id, t, _ in selected:\n        source = nodes_by_id[end_id]\n        target = nodes_by_id[start_id]\n        previous_id = end_id\n        inserted_ids: list[int] = []\n        for k in (1, 2):\n            frac = k / 3.0\n            mid_t = int(source["t"]) + k\n            midpoint = (\n                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,\n                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,\n                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,\n            )\n            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)\n            node_id = next_node_id\n            next_node_id += 1\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": mid_t,\n                "z": refined_point[0],\n                "y": refined_point[1],\n                "x": refined_point[2],\n            }\n            inserted_ids.append(node_id)\n            current = nodes_by_id[node_id]\n            new_edges.append({\n                "source_id": previous_id,\n                "target_id": node_id,\n                "edge_prob": None,\n                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),\n                "gap2_recovered": 1,\n            })\n            previous_id = node_id\n        new_edges.append({\n            "source_id": previous_id,\n            "target_id": start_id,\n            "edge_prob": None,\n            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),\n            "gap2_recovered": 1,\n        })\n        stats["gap2_pairs_selected"] += 1\n        stats["gap2_added_nodes"] += len(inserted_ids)\n        stats["gap2_added_edges"] += 3\n\n    return nodes_by_id, [*edges, *new_edges]\n\n\ndef add_safe_divisions_postlink(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:\n        return edges\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n\n    out_by_source: dict[int, list[dict[str, object]]] = {}\n    incoming: set[int] = set()\n    for edge in edges:\n        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)\n        incoming.add(int(edge["target_id"]))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n\n    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}\n    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))\n    added: list[dict[str, object]] = []\n    used_targets: set[int] = set()\n\n    for t in sorted(ids_by_t):\n        child_frame_ids = ids_by_t.get(t + 1, [])\n        if not child_frame_ids:\n            continue\n        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]\n        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]\n        if not source_ids or not candidate_ids:\n            continue\n\n        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))\n        proposals: list[tuple[float, int, int, float, float]] = []\n        for source_id in source_ids:\n            source = nodes_by_id[source_id]\n            existing_child_edge = out_by_source[source_id][0]\n            existing_child_id = int(existing_child_edge["target_id"])\n            existing_child = nodes_by_id.get(existing_child_id)\n            if existing_child is None or int(existing_child["t"]) != t + 1:\n                continue\n            child_dist = edge_distance_um(source, existing_child)\n            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:\n                continue\n            for candidate_id in candidate_ids:\n                if (source_id, candidate_id) in existing_edges:\n                    continue\n                candidate = nodes_by_id[candidate_id]\n                parent_dist = edge_distance_um(source, candidate)\n                if parent_dist > SAFE_DIV_MAX_UM:\n                    continue\n                sister_dist = edge_distance_um(existing_child, candidate)\n                if sister_dist > SAFE_DIV_SISTER_MAX_UM:\n                    continue\n                if DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(\n                    dataset,\n                    int(candidate["t"]),\n                    node_point(candidate),\n                    deepcenter_bundle,\n                    frame_cache,\n                    deepcenter_cache,\n                    stats,\n                    "safe_div",\n                    DEEPCENTER_SAFE_DIV_THRESHOLD,\n                ):\n                    continue\n                score = parent_dist + 0.15 * sister_dist\n                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\n\n        stats["safe_division_candidates"] += len(proposals)\n        if not proposals:\n            continue\n        proposals.sort(key=lambda item: item[0])\n        added_this_frame = 0\n        for _, source_id, candidate_id, parent_dist, _ in proposals:\n            if len(added) >= global_cap:\n                stats["safe_division_skipped_cap"] += 1\n                break\n            if added_this_frame >= frame_cap:\n                break\n            if candidate_id in used_targets or candidate_id in incoming:\n                continue\n            candidate = nodes_by_id[candidate_id]\n            added.append({\n                "source_id": source_id,\n                "target_id": candidate_id,\n                "edge_prob": None,\n                "distance_um": parent_dist,\n                "safe_division": 1,\n            })\n            used_targets.add(candidate_id)\n            added_this_frame += 1\n\n    if added:\n        stats["safe_divisions_added"] = len(added)\n        return [*edges, *added]\n    return edges\n\n\ndef filter_short_track_components(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:\n        return nodes_by_id, edges\n\n    parent = {node_id: node_id for node_id in nodes_by_id}\n\n    def find(node_id: int) -> int:\n        while parent[node_id] != node_id:\n            parent[node_id] = parent[parent[node_id]]\n            node_id = parent[node_id]\n        return node_id\n\n    def union(a: int, b: int) -> None:\n        if a not in parent or b not in parent:\n            return\n        ra = find(a)\n        rb = find(b)\n        if ra != rb:\n            parent[ra] = rb\n\n    out_count: dict[int, int] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        union(source_id, target_id)\n        out_count[source_id] = out_count.get(source_id, 0) + 1\n\n    components: dict[int, list[int]] = {}\n    for node_id in nodes_by_id:\n        components.setdefault(find(node_id), []).append(node_id)\n\n    component_edges: dict[int, list[dict[str, object]]] = {root: [] for root in components}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        if source_id in parent and target_id in parent:\n            component_edges.setdefault(find(source_id), []).append(edge)\n\n    keep: set[int] = set()\n    for root, members in components.items():\n        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)\n        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):\n            keep.update(members)\n\n    if not keep:\n        stats["short_track_filter_skipped_all"] += 1\n        return nodes_by_id, edges\n\n    removed_before_rescue = len(nodes_by_id) - len(keep)\n    if removed_before_rescue <= 0:\n        return nodes_by_id, edges\n\n    if ADAPTIVE_SHORT_TRACK_RESCUE:\n        removed_frac = removed_before_rescue / max(len(nodes_by_id), 1)\n        if removed_frac >= SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC:\n            budget = min(\n                SHORT_TRACK_RESCUE_MAX_NODES_ABS,\n                max(0, int(round(len(nodes_by_id) * SHORT_TRACK_RESCUE_MAX_NODES_FRAC))),\n            )\n            stats["short_track_rescue_triggered"] = 1\n            stats["short_track_rescue_budget"] = budget\n            proposals: list[tuple[float, int, float, int, list[int]]] = []\n            for root, members in components.items():\n                if set(members) & keep:\n                    continue\n                if len(members) < SHORT_TRACK_RESCUE_MIN_LEN or len(members) >= OUTPUT_MIN_TRACK_LEN:\n                    continue\n                c_edges = component_edges.get(root, [])\n                if not c_edges:\n                    continue\n                probs: list[float] = []\n                dists: list[float] = []\n                for edge in c_edges:\n                    try:\n                        prob = float(edge.get("edge_prob", 0.0))\n                    except (TypeError, ValueError):\n                        prob = 0.0\n                    if np.isfinite(prob):\n                        probs.append(prob)\n                    try:\n                        dist = float(edge.get("distance_um", np.nan))\n                    except (TypeError, ValueError):\n                        dist = np.nan\n                    if np.isfinite(dist):\n                        dists.append(dist)\n                mean_prob = float(np.mean(probs)) if probs else 0.0\n                mean_dist = float(np.mean(dists)) if dists else float("inf")\n                if mean_prob < SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB:\n                    continue\n                if mean_dist > SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM:\n                    continue\n                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)\n                proposals.append((score, len(members), mean_prob, root, members))\n            proposals.sort(reverse=True)\n            rescued_nodes = 0\n            rescued_components = 0\n            for _, size, _, _, members in proposals:\n                if budget <= 0 or rescued_nodes + size > budget:\n                    continue\n                keep.update(members)\n                rescued_nodes += size\n                rescued_components += 1\n            stats["short_track_rescue_components"] = rescued_components\n            stats["short_track_rescue_nodes"] = rescued_nodes\n\n    removed_nodes = len(nodes_by_id) - len(keep)\n    if removed_nodes <= 0:\n        return nodes_by_id, edges\n\n    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}\n    kept_edges = [\n        edge for edge in edges\n        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes\n    ]\n    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))\n    stats["short_track_nodes_removed"] = removed_nodes\n    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)\n    return kept_nodes, kept_edges\n\n\ndef linefit_smooth_output_graph(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> dict[int, dict[str, object]]:\n    """Smooth linear track interiors without changing graph topology."""\n    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:\n        return nodes_by_id\n\n    predecessor: dict[int, list[int]] = {}\n    successor: dict[int, list[int]] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        source = nodes_by_id.get(source_id)\n        target = nodes_by_id.get(target_id)\n        if source is None or target is None:\n            continue\n        if int(target["t"]) != int(source["t"]) + 1:\n            continue\n        successor.setdefault(source_id, []).append(target_id)\n        predecessor.setdefault(target_id, []).append(source_id)\n\n    original_pos = {\n        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)\n        for node_id, node in nodes_by_id.items()\n    }\n    updated_pos: dict[int, np.ndarray] = {}\n    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))\n\n    for node_id in sorted(nodes_by_id):\n        neighbourhood: list[tuple[int, int]] = [(0, node_id)]\n\n        current = node_id\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\n            prev_ids = predecessor.get(current, [])\n            if len(prev_ids) != 1:\n                break\n            current = prev_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((-step, current))\n\n        current = node_id\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\n            next_ids = successor.get(current, [])\n            if len(next_ids) != 1:\n                break\n            current = next_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((step, current))\n\n        if len(neighbourhood) < 3:\n            stats["linefit_skipped_nodes"] += 1\n            continue\n\n        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)\n        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])\n        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)\n        if not np.isfinite(fitted).all():\n            stats["linefit_skipped_nodes"] += 1\n            continue\n        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted\n\n    for node_id, pos in updated_pos.items():\n        nodes_by_id[node_id]["z"] = float(pos[0])\n        nodes_by_id[node_id]["y"] = float(pos[1])\n        nodes_by_id[node_id]["x"] = float(pos[2])\n\n    stats["linefit_smoothed_nodes"] = len(updated_pos)\n    return nodes_by_id\n\n\ndef filter_output_graph(\n    nodes_by_id: dict[int, dict[str, object]],\n    raw_edges: list[dict[str, object]],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:\n    stats = {\n        "raw_edges": len(raw_edges),\n        "dropped_nonconsecutive_edges": 0,\n        "dropped_long_edges": 0,\n        "dropped_multi_parent_edges": 0,\n        "dropped_multi_child_edges": 0,\n        "dropped_division_edges": 0,\n        "gap_candidates": 0,\n        "gap_pairs_selected": 0,\n        "gap_reused_existing": 0,\n        "gap_inserted_synthetic": 0,\n        "gap_added_nodes": 0,\n        "gap_added_edges": 0,\n        "gap_skipped_node_cap": 0,\n        "gap_refined_synthetic": 0,\n        "gap_refine_failed": 0,\n        "gap_refine_rejected_shift": 0,\n        "pruned_isolated_nodes": 0,\n        "motion_relink_edges": 0,\n        "motion_relink_tight_edges": 0,\n        "motion_relink_relaxed_edges": 0,\n        "motion_relink_frames": 0,\n        "motion_relink_replaced_raw_edges": 0,\n        "motion_relink_fallback_raw": 0,\n        "motion_relink_skipped_large_frame": 0,\n        "gap2_candidates": 0,\n        "gap2_pairs_selected": 0,\n        "gap2_added_nodes": 0,\n        "gap2_added_edges": 0,\n        "gap2_skipped_cap": 0,\n        "safe_division_candidates": 0,\n        "safe_divisions_added": 0,\n        "safe_division_skipped_cap": 0,\n        "deepcenter_gap_checked": 0,\n        "deepcenter_gap_accepted": 0,\n        "deepcenter_gap_rejected": 0,\n        "deepcenter_gap_missing": 0,\n        "deepcenter_safe_div_checked": 0,\n        "deepcenter_safe_div_accepted": 0,\n        "deepcenter_safe_div_rejected": 0,\n        "deepcenter_safe_div_missing": 0,\n        "short_track_components_removed": 0,\n        "short_track_nodes_removed": 0,\n        "short_track_edges_removed": 0,\n        "short_track_filter_skipped_all": 0,\n        "short_track_rescue_triggered": 0,\n        "short_track_rescue_components": 0,\n        "short_track_rescue_nodes": 0,\n        "short_track_rescue_budget": 0,\n        "linefit_smoothed_nodes": 0,\n        "linefit_skipped_nodes": 0,\n    }\n\n    edges: list[dict[str, object]] = []\n    for edge in raw_edges:\n        source = nodes_by_id.get(int(edge["source_id"]))\n        target = nodes_by_id.get(int(edge["target_id"]))\n        if source is None or target is None:\n            continue\n        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:\n            stats["dropped_nonconsecutive_edges"] += 1\n            continue\n        distance_um = edge_distance_um(source, target)\n        edge["distance_um"] = distance_um\n        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:\n            stats["dropped_long_edges"] += 1\n            continue\n        edges.append(edge)\n\n    if OUTPUT_MOTION_RELINK:\n        learned_edge_probs: dict[tuple[int, int], float] = {}\n        for edge in edges:\n            prob = edge.get("edge_prob")\n            if prob is None:\n                continue\n            try:\n                prob = float(prob)\n            except (TypeError, ValueError):\n                continue\n            if np.isfinite(prob):\n                key = (int(edge["source_id"]), int(edge["target_id"]))\n                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)\n        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs)\n        if motion_edges:\n            stats["motion_relink_replaced_raw_edges"] = len(edges)\n            edges = motion_edges\n        else:\n            stats["motion_relink_fallback_raw"] = 1\n\n    if OUTPUT_SINGLE_PARENT_REPAIR and edges:\n        best_by_target: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            target_id = int(edge["target_id"])\n            prev = best_by_target.get(target_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_target[target_id] = edge\n        kept_ids = {id(edge) for edge in best_by_target.values()}\n        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    if OUTPUT_SINGLE_CHILD_REPAIR and edges:\n        best_by_source: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            prev = best_by_source.get(source_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_source[source_id] = edge\n        kept_ids = {id(edge) for edge in best_by_source.values()}\n        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    repair_frame_cache: dict[int, np.ndarray] = {}\n    deepcenter_heatmap_cache: dict[tuple[str, int], np.ndarray] = {}\n    nodes_by_id, edges = close_single_frame_gaps(\n        nodes_by_id,\n        edges,\n        stats,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        frame_cache=repair_frame_cache,\n        deepcenter_cache=deepcenter_heatmap_cache,\n    )\n    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)\n    edges = add_safe_divisions_postlink(\n        nodes_by_id,\n        edges,\n        stats,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        frame_cache=repair_frame_cache,\n        deepcenter_cache=deepcenter_heatmap_cache,\n    )\n\n    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:\n        by_source: dict[int, list[dict[str, object]]] = {}\n        for edge in edges:\n            by_source.setdefault(int(edge["source_id"]), []).append(edge)\n\n        filtered: list[dict[str, object]] = []\n        for source_id, source_edges in by_source.items():\n            if len(source_edges) <= 1:\n                filtered.extend(source_edges)\n                continue\n\n            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)\n            source = nodes_by_id[source_id]\n            top1 = ranked[0]\n            top2 = ranked[1]\n            d1 = float(top1["distance_um"])\n            d2 = float(top2["distance_um"])\n            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])\n            valid_division = (\n                max(d1, d2) <= DIV_PARENT_MAX_UM\n                and sister <= DIV_SISTER_MAX_UM\n                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1\n                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1\n            )\n            if valid_division:\n                filtered.extend([top1, top2])\n                stats["dropped_division_edges"] += max(0, len(ranked) - 2)\n            elif DIV_DROP_TO_SINGLE_IF_BAD:\n                filtered.append(top1)\n                stats["dropped_division_edges"] += len(ranked) - 1\n            else:\n                filtered.extend(ranked)\n        edges = filtered\n\n    if OUTPUT_PRUNE_ISOLATED:\n        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}\n        if incident:\n            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}\n            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)\n            nodes_by_id = kept_nodes\n            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]\n\n    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)\n    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)\n\n    return nodes_by_id, edges, stats\n\n\nDEEPCENTER_VETO_DETECTOR = load_deepcenter_veto_detector()\n\ngeffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))\nprint(f"Found {len(geffs)} prediction graphs")\nif len(geffs) != len(test_stems):\n    found = {path.stem for path in geffs}\n    missing = sorted(set(test_stems) - found)\n    raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")\n\nstats_rows: list[dict[str, object]] = []\nseen_datasets: set[str] = set()\nrow_id = 0\ntotal_nodes = 0\ntotal_edges = 0\n\nwith SUBMISSION_PATH.open("w", newline="") as f:\n    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)\n    writer.writeheader()\n\n    for geff_path in geffs:\n        dataset = geff_path.stem\n        seen_datasets.add(dataset)\n        graph = graph_from_geff(geff_path)\n\n        nodes_by_id: dict[int, dict[str, object]] = {}\n        for row in graph.node_attrs().iter_rows(named=True):\n            node_id = int(row["node_id"])\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": int(row["t"]),\n                "z": float(row["z"]),\n                "y": float(row["y"]),\n                "x": float(row["x"]),\n            }\n\n        raw_edges: list[dict[str, object]] = []\n        for row in graph.edge_attrs().iter_rows(named=True):\n            edge_prob = row.get("edge_prob") if hasattr(row, "get") else None\n            raw_edges.append({\n                "source_id": int(row["source_id"]),\n                "target_id": int(row["target_id"]),\n                "edge_prob": None if edge_prob is None else float(edge_prob),\n            })\n\n        raw_node_count = len(nodes_by_id)\n        nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset, deepcenter_bundle=DEEPCENTER_VETO_DETECTOR)\n        if not nodes_by_id:\n            raise AssertionError(f"{dataset}: post-processing removed every node")\n\n        for node_id in sorted(nodes_by_id):\n            node = nodes_by_id[node_id]\n            writer.writerow({\n                "id": row_id,\n                "dataset": dataset,\n                "row_type": "node",\n                "node_id": int(node["node_id"]),\n                "t": int(node["t"]),\n                "z": max(0, int(round(float(node["z"])))),\n                "y": max(0, int(round(float(node["y"])))),\n                "x": max(0, int(round(float(node["x"])))),\n                "source_id": -1,\n                "target_id": -1,\n            })\n            row_id += 1\n\n        division_sources: dict[int, int] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            target_id = int(edge["target_id"])\n            if source_id not in nodes_by_id or target_id not in nodes_by_id:\n                raise AssertionError(f"{dataset}: dangling edge after filtering")\n            writer.writerow({\n                "id": row_id,\n                "dataset": dataset,\n                "row_type": "edge",\n                "node_id": -1,\n                "t": -1,\n                "z": -1,\n                "y": -1,\n                "x": -1,\n                "source_id": source_id,\n                "target_id": target_id,\n            })\n            row_id += 1\n            division_sources[source_id] = division_sources.get(source_id, 0) + 1\n\n        node_count = len(nodes_by_id)\n        edge_count = len(edges)\n        total_nodes += node_count\n        total_edges += edge_count\n        stats_rows.append({\n            "dataset": dataset,\n            "raw_nodes": raw_node_count,\n            "nodes": node_count,\n            "raw_edges": filter_stats["raw_edges"],\n            "edges": edge_count,\n            "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),\n            "edge_to_node_ratio": edge_count / max(node_count, 1),\n            "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),\n            **filter_stats,\n        })\n\nexpected_datasets = set(test_stems)\nmissing_datasets = sorted(expected_datasets - seen_datasets)\nextra_datasets = sorted(seen_datasets - expected_datasets)\nif missing_datasets or extra_datasets:\n    raise AssertionError({"missing": missing_datasets[:10], "extra": extra_datasets[:10]})\nassert row_id == total_nodes + total_edges, "Internal row counter mismatch"\nassert total_nodes > 0, "No node rows produced"\n\nheader = SUBMISSION_PATH.open().readline().strip().split(",")\nassert header == CSV_COLUMNS, f"Bad CSV header: {header}"\n\nstats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)\nstats["predict_minutes_total"] = predict_seconds / 60.0\nstats["experiment_tag"] = EXPERIMENT_TAG\nstats.to_csv(RUN_STATS_PATH, index=False)\n\nprint(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")\nprint(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")\nprint(f"Wrote {RUN_STATS_PATH}")\ndisplay(stats.describe(include="all"))\ndisplay(pd.read_csv(SUBMISSION_PATH, nrows=8))'
COPY_OUTPUT_CODE = "from pathlib import Path\nimport shutil\nimport pandas as pd\n\nsubmission_path = Path('submission.csv')\nrun_stats_path = Path('run_stats.csv')\n\nbatch_submission = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_submission.csv'\nbatch_stats = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_run_stats.csv'\nraw_backup = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_submission_raw_before_clip.csv'\nstable_submission = REPORT_DIR / 'deepcenter_local_validation_submission.csv'\nstable_stats = REPORT_DIR / 'deepcenter_local_validation_run_stats.csv'\n\nif not submission_path.exists():\n    raise FileNotFoundError(submission_path)\n\nshutil.copy2(submission_path, batch_submission)\nif not raw_backup.exists():\n    shutil.copy2(submission_path, raw_backup)\n\nsub = pd.read_csv(batch_submission)\nnodes = sub['row_type'].eq('node')\n\ndef count_bad_node_coords(df):\n    d = df.loc[df['row_type'].eq('node'), ['t', 'z', 'y', 'x']]\n    return int(((d['t'] < 0) | (d['t'] > 99) |\n                (d['z'] < 0) | (d['z'] > 63) |\n                (d['y'] < 0) | (d['y'] > 255) |\n                (d['x'] < 0) | (d['x'] > 255)).sum())\n\nbad_before = count_bad_node_coords(sub)\nsub.loc[nodes, 't'] = sub.loc[nodes, 't'].clip(0, 99)\nsub.loc[nodes, 'z'] = sub.loc[nodes, 'z'].clip(0, 63)\nsub.loc[nodes, 'y'] = sub.loc[nodes, 'y'].clip(0, 255)\nsub.loc[nodes, 'x'] = sub.loc[nodes, 'x'].clip(0, 255)\n\nint_cols = ['id', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']\nsub[int_cols] = sub[int_cols].astype('int64')\nsub.to_csv(batch_submission, index=False)\nbad_after = count_bad_node_coords(sub)\n\nif run_stats_path.exists():\n    shutil.copy2(run_stats_path, batch_stats)\n\nif WRITE_STABLE_ALIAS:\n    shutil.copy2(batch_submission, stable_submission)\n    if batch_stats.exists():\n        shutil.copy2(batch_stats, stable_stats)\n\nprint('raw backup:', raw_backup)\nprint('bad node coords before clip:', bad_before)\nprint('bad node coords after clip:', bad_after)\nprint('targeted clipped submission:', batch_submission)\nprint('targeted run stats:', batch_stats if batch_stats.exists() else 'missing')\nprint('WRITE_STABLE_ALIAS:', WRITE_STABLE_ALIAS)\nprint('Now run the local sparse-GT score cell below.')\n"
SCORE_CODE = "from pathlib import Path\nimport json\nimport math\n\nimport numpy as np\nimport pandas as pd\nimport zarr\nfrom scipy.optimize import linear_sum_assignment\nfrom tqdm.auto import tqdm\n\nMATCH_MAX_DISTANCE_UM = 7.0\nVOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=float)\n\nsubmission_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_submission.csv'\nif not submission_path.exists():\n    local_submission = Path('submission.csv')\n    if local_submission.exists():\n        submission_path = local_submission\n    else:\n        raise FileNotFoundError('No targeted DeepCenter submission CSV found. Run build + copy cells first.')\n\nprint('loading submission:', submission_path)\nsubmission_df = pd.read_csv(submission_path)\nscore_sample_ids = sorted(submission_df['dataset'].astype(str).unique())\nvalidation_df = pd.DataFrame({'sample_id': score_sample_ids})\nvalidation_df['embryo_id'] = validation_df['sample_id'].str.split('_').str[0]\nvalidation_df['geff_path'] = validation_df['sample_id'].map(lambda s: BASE_DIR / 'train' / f'{s}.geff')\nvalidation_df['has_geff'] = validation_df['geff_path'].map(lambda p: p.exists())\n\nnode_rows = submission_df['row_type'].eq('node')\nbad_node_coords = int((\n    (submission_df.loc[node_rows, 't'] < 0) | (submission_df.loc[node_rows, 't'] > 99) |\n    (submission_df.loc[node_rows, 'z'] < 0) | (submission_df.loc[node_rows, 'z'] > 63) |\n    (submission_df.loc[node_rows, 'y'] < 0) | (submission_df.loc[node_rows, 'y'] > 255) |\n    (submission_df.loc[node_rows, 'x'] < 0) | (submission_df.loc[node_rows, 'x'] > 255)\n).sum())\n\nprint('score samples:', len(validation_df))\nprint('submission rows:', len(submission_df))\nprint('node rows:', int(node_rows.sum()))\nprint('edge rows:', int((submission_df['row_type'] == 'edge').sum()))\nprint('bad node coordinate rows:', bad_node_coords)\ndisplay(validation_df[['sample_id', 'embryo_id', 'has_geff']].head(20))\nassert validation_df['has_geff'].all(), 'Some validation GEFF files are missing'\nassert bad_node_coords == 0, 'Submission still has out-of-bound node coordinates. Run the copy/clip cell first.'\n\nsubmission_groups = submission_df.groupby('dataset', sort=False)\n\n\ndef read_geff(geff_path: Path):\n    root = zarr.open(str(geff_path), mode='r')\n    node_ids = np.asarray(root['nodes/ids'][:]).astype(int)\n    nodes = pd.DataFrame({\n        'node_id': node_ids,\n        't': np.asarray(root['nodes/props/t/values'][:]).astype(int),\n        'z': np.asarray(root['nodes/props/z/values'][:]).astype(float),\n        'y': np.asarray(root['nodes/props/y/values'][:]).astype(float),\n        'x': np.asarray(root['nodes/props/x/values'][:]).astype(float),\n    })\n    edge_ids = np.asarray(root['edges/ids'][:]).astype(int)\n    edges = pd.DataFrame(edge_ids, columns=['source_id', 'target_id']) if edge_ids.size else pd.DataFrame(columns=['source_id', 'target_id'])\n\n    estimated = np.nan\n    meta_path = geff_path / 'zarr.json'\n    if meta_path.exists():\n        try:\n            meta = json.loads(meta_path.read_text())\n            attrs = meta.get('attributes', {})\n            estimated = attrs.get('estimated_number_of_nodes', np.nan)\n        except Exception:\n            pass\n    return nodes, edges, estimated\n\n\ndef physical_distance_matrix_um(a_zyx, b_zyx):\n    a = np.asarray(a_zyx, dtype=float) * VOXEL_SCALE_UM\n    b = np.asarray(b_zyx, dtype=float) * VOXEL_SCALE_UM\n    diff = a[:, None, :] - b[None, :, :]\n    return np.sqrt((diff * diff).sum(axis=2))\n\n\ndef match_pred_to_gt_by_time(pred_nodes, gt_nodes, max_distance_um=MATCH_MAX_DISTANCE_UM):\n    rows = []\n    pred_to_gt = {}\n    gt_to_pred = {}\n    common_times = sorted(set(pred_nodes['t']).intersection(set(gt_nodes['t'])))\n    for t in common_times:\n        pred_t = pred_nodes[pred_nodes['t'] == t].reset_index(drop=True)\n        gt_t = gt_nodes[gt_nodes['t'] == t].reset_index(drop=True)\n        if pred_t.empty or gt_t.empty:\n            continue\n        dist = physical_distance_matrix_um(pred_t[['z', 'y', 'x']].to_numpy(), gt_t[['z', 'y', 'x']].to_numpy())\n        rr, cc = linear_sum_assignment(dist)\n        for r, c in zip(rr, cc):\n            d = float(dist[r, c])\n            if d <= max_distance_um:\n                pred_node_id = int(pred_t.loc[r, 'node_id'])\n                gt_node_id = int(gt_t.loc[c, 'node_id'])\n                rows.append({'t': int(t), 'pred_node_id': pred_node_id, 'gt_node_id': gt_node_id, 'distance_um': d})\n                pred_to_gt[pred_node_id] = gt_node_id\n                gt_to_pred[gt_node_id] = pred_node_id\n    return pd.DataFrame(rows), pred_to_gt, gt_to_pred\n\n\ndef split_submission_group(sample_df):\n    nodes = sample_df[sample_df['row_type'] == 'node'][['node_id', 't', 'z', 'y', 'x']].copy()\n    edges = sample_df[sample_df['row_type'] == 'edge'][['source_id', 'target_id']].copy()\n    return nodes, edges\n\n\ndef division_diagnostics(pred_edges, gt_edges, pred_to_gt):\n    if len(gt_edges):\n        gt_out = gt_edges['source_id'].astype(int).value_counts()\n        gt_div_nodes = set(gt_out[gt_out >= 2].index.astype(int))\n    else:\n        gt_div_nodes = set()\n\n    if len(pred_edges):\n        pred_out = pred_edges['source_id'].astype(int).value_counts()\n        pred_div_nodes = set(pred_out[pred_out >= 2].index.astype(int))\n    else:\n        pred_div_nodes = set()\n\n    matched_pred_div_gt = {pred_to_gt[p] for p in pred_div_nodes if p in pred_to_gt}\n    direct_tp = len(gt_div_nodes & matched_pred_div_gt)\n    direct_fp = max(0, len(pred_div_nodes) - direct_tp)\n    direct_fn = max(0, len(gt_div_nodes) - direct_tp)\n    direct_j = direct_tp / max(1, direct_tp + direct_fp + direct_fn)\n\n    return {\n        'gt_divisions_direct': int(len(gt_div_nodes)),\n        'pred_divisions_direct': int(len(pred_div_nodes)),\n        'division_tp_direct_approx': int(direct_tp),\n        'division_fp_direct_approx': int(direct_fp),\n        'division_fn_direct_approx': int(direct_fn),\n        'division_jaccard_direct_approx': float(direct_j),\n    }\n\n\ndef score_one_sample(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_number_of_nodes=np.nan):\n    matches, pred_to_gt, gt_to_pred = match_pred_to_gt_by_time(pred_nodes, gt_nodes)\n    gt_edge_set = set(map(tuple, gt_edges[['source_id', 'target_id']].astype(int).to_numpy())) if len(gt_edges) else set()\n\n    edge_tp = 0\n    matched_pred_edges = 0\n    for row in pred_edges[['source_id', 'target_id']].astype(int).itertuples(index=False):\n        src_gt = pred_to_gt.get(int(row.source_id))\n        tgt_gt = pred_to_gt.get(int(row.target_id))\n        if src_gt is None or tgt_gt is None:\n            continue\n        matched_pred_edges += 1\n        if (src_gt, tgt_gt) in gt_edge_set:\n            edge_tp += 1\n\n    edge_fp_matched = max(0, matched_pred_edges - edge_tp)\n    edge_fn = max(0, len(gt_edge_set) - edge_tp)\n    edge_den_matched = max(1, edge_tp + edge_fp_matched + edge_fn)\n    edge_jaccard = edge_tp / edge_den_matched\n    edge_jaccard_naive = edge_tp / max(1, edge_tp + max(0, len(pred_edges) - edge_tp) + edge_fn)\n\n    if np.isfinite(estimated_number_of_nodes) and estimated_number_of_nodes > 0:\n        over_factor = min(1.0, float(estimated_number_of_nodes) / max(1, len(pred_nodes)))\n    else:\n        over_factor = 1.0\n\n    result = {\n        'pred_nodes': int(len(pred_nodes)),\n        'gt_nodes_sparse': int(len(gt_nodes)),\n        'matched_nodes': int(len(matches)),\n        'node_recall_sparse_gt': float(len(matches) / max(1, len(gt_nodes))),\n        'node_precision_sparse_gt': float(len(matches) / max(1, len(pred_nodes))),\n        'pred_edges': int(len(pred_edges)),\n        'gt_edges_sparse': int(len(gt_edge_set)),\n        'matched_pred_edges': int(matched_pred_edges),\n        'edge_tp': int(edge_tp),\n        'edge_fp_matched': int(edge_fp_matched),\n        'edge_fn': int(edge_fn),\n        'edge_den_matched': int(edge_den_matched),\n        'edge_jaccard_matched': float(edge_jaccard),\n        'edge_jaccard_naive': float(edge_jaccard_naive),\n        'estimated_number_of_nodes': float(estimated_number_of_nodes) if np.isfinite(estimated_number_of_nodes) else np.nan,\n        'node_overprediction_factor': float(over_factor),\n        'edge_jaccard_adjusted_approx': float(edge_jaccard * over_factor),\n        'match_distance_median_um': float(matches['distance_um'].median()) if len(matches) else np.nan,\n        'match_distance_p95_um': float(matches['distance_um'].quantile(0.95)) if len(matches) else np.nan,\n    }\n    result.update(division_diagnostics(pred_edges, gt_edges, pred_to_gt))\n    return result\n\nscore_rows = []\nfor row in tqdm(validation_df.itertuples(index=False), total=len(validation_df), desc='local sparse-GT score'):\n    sample_id = row.sample_id\n    gt_nodes, gt_edges, estimated = read_geff(Path(row.geff_path))\n    try:\n        sample_submission = submission_groups.get_group(sample_id)\n    except KeyError:\n        print('missing predictions for', sample_id)\n        continue\n    pred_nodes, pred_edges = split_submission_group(sample_submission)\n    if pred_nodes.empty:\n        print('missing node predictions for', sample_id)\n        continue\n    score = score_one_sample(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated)\n    score.update({'method': 'deepcenter', 'sample_id': sample_id, 'embryo_id': row.embryo_id})\n    score_rows.append(score)\n\nscores_df = pd.DataFrame(score_rows)\nif scores_df.empty:\n    raise RuntimeError('No scores were computed.')\n\nweighted_den = scores_df['edge_den_matched'].clip(lower=1)\nweighted_edge = float((scores_df['edge_jaccard_matched'] * weighted_den).sum() / weighted_den.sum())\nweighted_adjusted = float((scores_df['edge_jaccard_adjusted_approx'] * weighted_den).sum() / weighted_den.sum())\n\nsummary_rows = [{\n    'method': 'deepcenter',\n    'n_samples': int(len(scores_df)),\n    'local_weighted_edge_jaccard_approx': weighted_edge,\n    'local_weighted_adjusted_edge_jaccard_approx': weighted_adjusted,\n    'mean_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].mean()),\n    'median_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].median()),\n    'p10_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].quantile(0.10)),\n    'min_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].min()),\n    'mean_node_recall_sparse_gt': float(scores_df['node_recall_sparse_gt'].mean()),\n    'total_edge_tp': int(scores_df['edge_tp'].sum()),\n    'total_edge_fp_matched': int(scores_df['edge_fp_matched'].sum()),\n    'total_edge_fn': int(scores_df['edge_fn'].sum()),\n    'total_gt_divisions_direct': int(scores_df['gt_divisions_direct'].sum()),\n    'total_pred_divisions_direct': int(scores_df['pred_divisions_direct'].sum()),\n    'direct_division_jaccard_diagnostic': float(\n        scores_df['division_tp_direct_approx'].sum() /\n        max(1, scores_df['division_tp_direct_approx'].sum() + scores_df['division_fp_direct_approx'].sum() + scores_df['division_fn_direct_approx'].sum())\n    ),\n}]\nsummary_df = pd.DataFrame(summary_rows)\nby_embryo_df = scores_df.groupby('embryo_id').agg(\n    n_samples=('sample_id', 'count'),\n    mean_edge_jaccard_matched=('edge_jaccard_matched', 'mean'),\n    median_edge_jaccard_matched=('edge_jaccard_matched', 'median'),\n    min_edge_jaccard_matched=('edge_jaccard_matched', 'min'),\n    mean_node_recall_sparse_gt=('node_recall_sparse_gt', 'mean'),\n    mean_pred_nodes=('pred_nodes', 'mean'),\n    mean_pred_edges=('pred_edges', 'mean'),\n).reset_index()\nworst_df = scores_df.sort_values('edge_jaccard_matched').head(25).copy()\n\nscore_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_scores.csv'\nsummary_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_score_summary.csv'\nworst_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_worst_samples.csv'\nby_embryo_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_by_embryo.csv'\nstable_score_path = REPORT_DIR / 'deepcenter_local_validation_scores.csv'\nstable_summary_path = REPORT_DIR / 'deepcenter_local_validation_score_summary.csv'\nstable_worst_path = REPORT_DIR / 'deepcenter_local_validation_worst_samples.csv'\n\nscores_df.to_csv(score_path, index=False)\nsummary_df.to_csv(summary_path, index=False)\nworst_df.to_csv(worst_path, index=False)\nby_embryo_df.to_csv(by_embryo_path, index=False)\nscores_df.to_csv(stable_score_path, index=False)\nsummary_df.to_csv(stable_summary_path, index=False)\nworst_df.to_csv(stable_worst_path, index=False)\n\nprint('LOCAL SCORE SUMMARY')\ndisplay(summary_df)\nprint('BY EMBRYO')\ndisplay(by_embryo_df)\nprint('PER-SAMPLE SCORES')\ndisplay(scores_df)\nprint('WORST SAMPLES')\ndisplay(worst_df)\n\nprint('saved scores:', score_path)\nprint('saved summary:', summary_path)\nprint('saved worst:', worst_path)\nprint('stable summary:', stable_summary_path)\nprint('Note: direct division diagnostic is intentionally conservative and is not the official Kaggle division metric.')\n"

# Exact d3ac1a preset values are restored before every variant.
BASE_PRESET_ENV = {
    'BIOHUB_OUTPUT_FILTER_SHORT_TRACKS': '1',
    'BIOHUB_DET_THRESHOLD': '0.97',
    'BIOHUB_GAP_CLOSE_MAX_GAP': '2',
    'BIOHUB_OUTPUT_MIN_TRACK_LEN': '6',
    'BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS': '1',
    'BIOHUB_OUTPUT_GAP2_RECOVERY': '0',
    'BIOHUB_SAFE_DIV_MAX_UM': '4.66',
    'BIOHUB_SAFE_DIV_SISTER_MAX_UM': '7.05',
    'BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM': '7.65',
    'BIOHUB_SAFE_DIV_FRAME_FRAC_CAP': '0.0076',
    'BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP': '0.00375',
    'BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE': '1',
    'BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC': '0.10',
    'BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN': '4',
    'BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB': '0.82',
    'BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM': '3.25',
    'BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC': '0.018',
    'BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS': '180',
    'BIOHUB_USE_DEEPCENTER_VETO': '0',
    'BIOHUB_DEEPCENTER_GAP_VETO': '0',
    'BIOHUB_DEEPCENTER_SAFE_DIV_VETO': '0',
    'BIOHUB_DEEPCENTER_GAP_THRESHOLD': '0.06',
    'BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD': '0.08',
}

SHORT_TRACK_VARIANTS = {
    'd3ac1a_exact_min6_adaptive': {},
    'd3ac1a_global_min5_adaptive': {
        'BIOHUB_OUTPUT_MIN_TRACK_LEN': '5',
    },
    'd3ac1a_global_min5_no_adaptive': {
        'BIOHUB_OUTPUT_MIN_TRACK_LEN': '5',
        'BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE': '0',
    },
    'd3ac1a_min6_adaptive_expanded': {
        'BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC': '0.08',
        'BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC': '0.025',
        'BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS': '300',
    },
}

VARIANT_ENV_KEYS = sorted(set(BASE_PRESET_ENV) | {
    key for values in SHORT_TRACK_VARIANTS.values() for key in values
})
WRITE_STABLE_ALIAS = False


def activate_variant(name):
    for key in VARIANT_ENV_KEYS:
        os.environ.pop(key, None)
    for key, value in BASE_PRESET_ENV.items():
        os.environ[key] = value
    for key, value in SHORT_TRACK_VARIANTS[name].items():
        os.environ[key] = value
    global OUTPUT_PREFIX
    OUTPUT_PREFIX = f'd3ac1a_worst8_guard4_{name}'
    print()
    print('=' * 92)
    print('ACTIVE VARIANT:', name)
    print('OUTPUT_PREFIX:', OUTPUT_PREFIX)
    print('overrides:', json.dumps(SHORT_TRACK_VARIANTS[name], indent=2))


## Build, Persist, and Score Every Variant

Every completed variant writes its submission, run statistics, per-sample scores, and progress CSV to Drive before the next variant starts.

In [ ]:
ablation_rows = []
reference_row = None
progress_path = REPORT_DIR / 'd3ac1a_worst8_guard4_shorttrack_ablation_progress.csv'

variant_bar = tqdm(list(SHORT_TRACK_VARIANTS), desc='d3ac1a short-track variants', unit='variant')
for variant_index, variant_name in enumerate(variant_bar, start=1):
    variant_bar.set_postfix_str(variant_name)
    variant_start = time.time()
    activate_variant(variant_name)

    stage_bar = tqdm(['config', 'build_csv', 'copy_clip', 'score'], desc=f'{variant_name} stages', leave=False)
    stage_bar.set_postfix_str('config')
    exec(CONFIG_CODE, globals())
    stage_bar.update(1)

    stage_bar.set_postfix_str('build submission directly on Drive')
    exec(BUILD_SUBMISSION_CODE, globals())
    stage_bar.update(1)

    stage_bar.set_postfix_str('persist + clip on Drive')
    exec(COPY_OUTPUT_CODE, globals())
    stage_bar.update(1)

    stage_bar.set_postfix_str('local sparse-GT score')
    exec(SCORE_CODE, globals())
    stage_bar.update(1)
    stage_bar.close()

    summary_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_score_summary.csv'
    scores_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_scores.csv'
    stats_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_run_stats.csv'
    summary = pd.read_csv(summary_path).iloc[0]
    variant_scores = pd.read_csv(scores_path)
    variant_stats = pd.read_csv(stats_path)

    def weighted_for(ids):
        part = variant_scores[variant_scores['sample_id'].astype(str).isin(ids)]
        den = part['edge_tp'].sum() + part['edge_fp_matched'].sum() + part['edge_fn'].sum()
        return float(part['edge_tp'].sum() / max(1, den))

    row = {
        'variant': variant_name,
        'variant_index': variant_index,
        'n_samples': int(summary['n_samples']),
        'weighted_edge': float(summary['local_weighted_edge_jaccard_approx']),
        'worst8_weighted_edge': weighted_for(worst8_ids),
        'guard4_weighted_edge': weighted_for(guard4_ids),
        'mean_edge': float(summary['mean_edge_jaccard_matched']),
        'mean_node_recall': float(summary['mean_node_recall_sparse_gt']),
        'edge_tp': int(summary['total_edge_tp']),
        'edge_fp': int(summary['total_edge_fp_matched']),
        'edge_fn': int(summary['total_edge_fn']),
        'pred_divisions_direct': int(summary['total_pred_divisions_direct']),
        'short_track_nodes_removed': int(variant_stats.get('short_track_nodes_removed', pd.Series(dtype=int)).sum()),
        'short_track_rescue_nodes': int(variant_stats.get('short_track_rescue_nodes', pd.Series(dtype=int)).sum()),
        'short_track_rescue_components': int(variant_stats.get('short_track_rescue_components', pd.Series(dtype=int)).sum()),
        'elapsed_min': round((time.time() - variant_start) / 60, 2),
    }
    if reference_row is None:
        reference_row = row.copy()

        # Cross-pipeline audit: old 0.889 pipeline vs exact new d3ac1a 0.900 pipeline
        # on the same samples, GT, matching radius, and local scorer.
        compare_cols = [
            'sample_id', 'edge_jaccard_matched', 'node_recall_sparse_gt',
            'edge_tp', 'edge_fp_matched', 'edge_fn',
        ]
        old_selected = scores_all[scores_all['sample_id'].astype(str).isin(sample_ids)][compare_cols].copy()
        cross_pipeline = old_selected.merge(
            variant_scores[compare_cols], on='sample_id', suffixes=('_old0899', '_d3ac1a'), validate='one_to_one'
        )
        cross_pipeline['group'] = cross_pipeline['sample_id'].map(
            lambda sid: 'guard4' if str(sid) in set(guard4_ids) else 'old_worst8'
        )
        cross_pipeline['delta_edge_d3ac1a_vs_old'] = (
            cross_pipeline['edge_jaccard_matched_d3ac1a'] - cross_pipeline['edge_jaccard_matched_old0899']
        )
        cross_pipeline['delta_recall_d3ac1a_vs_old'] = (
            cross_pipeline['node_recall_sparse_gt_d3ac1a'] - cross_pipeline['node_recall_sparse_gt_old0899']
        )
        cross_pipeline['delta_tp_d3ac1a_vs_old'] = cross_pipeline['edge_tp_d3ac1a'] - cross_pipeline['edge_tp_old0899']
        cross_pipeline['delta_fp_d3ac1a_vs_old'] = cross_pipeline['edge_fp_matched_d3ac1a'] - cross_pipeline['edge_fp_matched_old0899']
        cross_pipeline['delta_fn_d3ac1a_vs_old'] = cross_pipeline['edge_fn_d3ac1a'] - cross_pipeline['edge_fn_old0899']

        cross_path = REPORT_DIR / 'd3ac1a_exact_vs_old0899_worst8_guard4_per_sample.csv'
        cross_pipeline.to_csv(cross_path, index=False)

        cross_summary_rows = []
        for group_name, group_df in [('all12', cross_pipeline), *list(cross_pipeline.groupby('group'))]:
            old_tp = int(group_df['edge_tp_old0899'].sum())
            old_fp = int(group_df['edge_fp_matched_old0899'].sum())
            old_fn = int(group_df['edge_fn_old0899'].sum())
            new_tp = int(group_df['edge_tp_d3ac1a'].sum())
            new_fp = int(group_df['edge_fp_matched_d3ac1a'].sum())
            new_fn = int(group_df['edge_fn_d3ac1a'].sum())
            old_weighted = old_tp / max(1, old_tp + old_fp + old_fn)
            new_weighted = new_tp / max(1, new_tp + new_fp + new_fn)
            cross_summary_rows.append({
                'group': group_name,
                'n_samples': len(group_df),
                'old0899_weighted_edge': old_weighted,
                'd3ac1a_weighted_edge': new_weighted,
                'delta_weighted_d3ac1a_vs_old': new_weighted - old_weighted,
                'delta_tp': new_tp - old_tp,
                'delta_fp': new_fp - old_fp,
                'delta_fn': new_fn - old_fn,
                'improved_samples': int((group_df['delta_edge_d3ac1a_vs_old'] > 1e-12).sum()),
                'same_samples': int((group_df['delta_edge_d3ac1a_vs_old'].abs() <= 1e-12).sum()),
                'worse_samples': int((group_df['delta_edge_d3ac1a_vs_old'] < -1e-12).sum()),
            })
        cross_summary = pd.DataFrame(cross_summary_rows)
        cross_summary_path = REPORT_DIR / 'd3ac1a_exact_vs_old0899_worst8_guard4_summary.csv'
        cross_summary.to_csv(cross_summary_path, index=False)
        print('CROSS-PIPELINE AUDIT: old 0.889 vs exact d3ac1a 0.900')
        display(cross_summary)
        display(cross_pipeline.sort_values('delta_edge_d3ac1a_vs_old'))
        print('saved cross-pipeline per-sample:', cross_path)
        print('saved cross-pipeline summary:', cross_summary_path)

    row['reference_weighted_edge'] = reference_row['weighted_edge']
    row['delta_weighted_vs_reference'] = row['weighted_edge'] - reference_row['weighted_edge']
    row['delta_worst8_vs_reference'] = row['worst8_weighted_edge'] - reference_row['worst8_weighted_edge']
    row['delta_guard4_vs_reference'] = row['guard4_weighted_edge'] - reference_row['guard4_weighted_edge']
    row['delta_tp_vs_reference'] = row['edge_tp'] - reference_row['edge_tp']
    row['delta_fp_vs_reference'] = row['edge_fp'] - reference_row['edge_fp']
    row['delta_fn_vs_reference'] = row['edge_fn'] - reference_row['edge_fn']
    ablation_rows.append(row)

    # Persist after every completed variant so a runtime loss cannot erase finished experiments.
    progress_df = pd.DataFrame(ablation_rows)
    progress_df.to_csv(progress_path, index=False)
    print('saved progress:', progress_path)
    display(progress_df.sort_values('weighted_edge', ascending=False))

ablation_df = pd.DataFrame(ablation_rows).sort_values('weighted_edge', ascending=False)
final_path = REPORT_DIR / 'd3ac1a_worst8_guard4_shorttrack_ablation_final.csv'
ablation_df.to_csv(final_path, index=False)
print('saved final comparison:', final_path)
display(ablation_df)
